<a href="https://colab.research.google.com/github/clee2026/MSDS_498/blob/main/capstone/chatbot/nyc311_genai_call_deflection_chatbot_colab_enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NYC 311 GenAI Call-Deflection Chatbot Prototype

## Purpose

This notebook builds a GenAI-style prototype that can help offload some common NYC 311 calls by:

- interpreting a user's issue
- matching it to common 311 complaint patterns
- using project outputs from the NYC 311 analysis
- querying the NYC Open Data API for recent/historical complaint patterns
- generating a response with an OpenAI API call

## Required Uploads

Before running this notebook in Colab, upload:

- `notebook2_outputs.zip`
- `notebook3_outputs.zip`

Recommended optional CSV files:

- `top_problem_counts.csv`
- `borough_performance.csv`
- `agency_performance.csv`
- `repeat_complaint_candidates.csv`
- `kpi_summary.csv`
- `channel_usage.csv`
- `delay_rate_by_problem.csv`
- `top_tokens_resolution_description.csv`

## Required API Key

You need an OpenAI API key. The notebook will ask for it using `getpass`, so it is not printed to the notebook output.

## Scope Note

This notebook creates a GenAI-style NYC 311 call-deflection chatbot prototype.

It combines:

- OpenAI API for natural-language understanding and response generation
- NYC Open Data API for live/historical 311 pattern lookup
- project outputs from Notebook 2 and Notebook 3
- prepared sample parquet for local dataset context

This prototype can support common informational 311 questions, but it does not submit official service requests or check official service request status.

This chatbot does not submit real 311 requests and does not check live request status. It is a call-deflection prototype that provides guidance, routing suggestions, and expected patterns based on public 311 data and project outputs.



In [1]:
# 1. Install and Import Libraries

!pip -q install openai pandas pyarrow requests tabulate

import os
import json
import zipfile
from pathlib import Path
from getpass import getpass
from typing import Dict, Optional

import requests
import pandas as pd
import numpy as np

from google.colab import files


## Upload project output files

Upload `notebook2_outputs.zip`, `notebook3_outputs.zip`, and optional CSV outputs when prompted.

These datasets derivied from EDA and initial findings reports that contain statistical data and information from the 20M parquet records act as a backup data source when the NYC311 API queries are not functional due to lack of connectivity or other restrictions.

In [3]:
# 2. Upload Files

uploaded = files.upload()

DATA_DIR = Path("/content/nyc311_chatbot_data")
EXTRACT_DIR = DATA_DIR / "extracted"
DATA_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)

for filename in uploaded.keys():
    src = Path("/content") / filename
    dst = DATA_DIR / filename
    if src.exists():
        if dst.exists():
            dst.unlink()
        src.rename(dst)

print("Uploaded files moved to:", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)


Saving agency_performance.csv to agency_performance.csv
Saving borough_performance.csv to borough_performance.csv
Saving channel_usage.csv to channel_usage.csv
Saving delay_rate_by_problem.csv to delay_rate_by_problem.csv
Saving kpi_summary.csv to kpi_summary.csv
Saving repeat_complaint_candidates.csv to repeat_complaint_candidates.csv
Saving top_problem_counts.csv to top_problem_counts.csv
Saving top_tokens_resolution_description.csv to top_tokens_resolution_description.csv
Uploaded files moved to: /content/nyc311_chatbot_data
 - agency_performance.csv
 - borough_performance.csv
 - channel_usage.csv
 - delay_rate_by_problem.csv
 - extracted
 - kpi_summary.csv
 - notebook2_outputs.zip
 - notebook3_outputs.zip
 - repeat_complaint_candidates.csv
 - top_problem_counts.csv
 - top_tokens_resolution_description.csv


In [4]:
# 3. Extract ZIP Files and Locate Files

for zip_path in DATA_DIR.glob("*.zip"):
    out_dir = EXTRACT_DIR / zip_path.stem
    out_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print("Extracted:", zip_path.name, "->", out_dir)

def find_file(filename: str) -> Optional[Path]:
    search_roots = [DATA_DIR, EXTRACT_DIR]
    for root in search_roots:
        direct = root / filename
        if direct.exists():
            return direct
    for root in search_roots:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None


Extracted: notebook2_outputs.zip -> /content/nyc311_chatbot_data/extracted/notebook2_outputs
Extracted: notebook3_outputs.zip -> /content/nyc311_chatbot_data/extracted/notebook3_outputs


## Configure OpenAI API

Paste your OpenAI API key when prompted. It will be stored only in the current Colab runtime.

In [5]:
# 4. Configure OpenAI API

from openai import OpenAI

api_key = getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# You can change this model if needed.
OPENAI_MODEL = "gpt-4.1-mini"

print("OpenAI client configured.")


Paste your OpenAI API key: ··········
OpenAI client configured.


## Load local project outputs

This step loads the project tables and the prepared sample parquet from Notebook 2.

In [6]:
# 5. Load Local Project Outputs

OPTIONAL_CSVS = [
    "top_problem_counts.csv",
    "borough_performance.csv",
    "agency_performance.csv",
    "repeat_complaint_candidates.csv",
    "kpi_summary.csv",
    "channel_usage.csv",
    "delay_rate_by_problem.csv",
    "top_tokens_resolution_description.csv",
    "top_findings_table.csv",
    "model_metrics.csv",
    "regression_model_metrics.csv",
    "classification_model_metrics.csv",
    "full_counts_complaint_type.csv",
    "full_counts_borough.csv",
    "full_counts_agency.csv",
    "full_monthly_counts.csv",
    "sample_response_time_summary.csv",
    "sample_response_by_complaint_type.csv",
]

tables = {}

for fname in OPTIONAL_CSVS:
    path = find_file(fname)
    if path:
        try:
            tables[fname] = pd.read_csv(path)
            print("Loaded:", fname, tables[fname].shape)
        except Exception as e:
            print("Could not load", fname, ":", e)

sample_path = find_file("analysis_ready_sample.parquet")
if sample_path:
    df_sample = pd.read_parquet(sample_path)
    print("Loaded sample parquet:", df_sample.shape)
else:
    df_sample = pd.DataFrame()
    print("No analysis_ready_sample.parquet found.")


Loaded: top_problem_counts.csv (192, 2)
Loaded: borough_performance.csv (6, 4)
Loaded: agency_performance.csv (15, 4)
Loaded: repeat_complaint_candidates.csv (170, 2)
Loaded: kpi_summary.csv (8, 2)
Loaded: channel_usage.csv (5, 2)
Loaded: delay_rate_by_problem.csv (192, 3)
Loaded: top_tokens_resolution_description.csv (30, 2)
Loaded: top_findings_table.csv (5, 5)
Loaded: model_metrics.csv (6, 11)
Loaded: regression_model_metrics.csv (3, 7)
Loaded: classification_model_metrics.csv (3, 8)
Loaded: full_counts_complaint_type.csv (25, 2)
Loaded: full_counts_borough.csv (7, 2)
Loaded: full_counts_agency.csv (20, 2)
Loaded: full_monthly_counts.csv (76, 2)
Loaded: sample_response_time_summary.csv (10, 2)
Loaded: sample_response_by_complaint_type.csv (25, 4)
Loaded sample parquet: (150000, 39)


## NYC Open Data API helper

This uses the NYC Open Data Socrata API for the 311 Service Requests dataset.

In [7]:
# 6. NYC Open Data API Helpers - Fast / Safe Version

import requests
import pandas as pd
from typing import Dict

NYC_311_API = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

def nyc_api_get(params: Dict, timeout: int = 30) -> pd.DataFrame:
    """
    Small, safe NYC Open Data API request.
    Designed for demo use only, not large analysis.
    """
    try:
        print("Calling NYC API...")
        r = requests.get(NYC_311_API, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        print(f"NYC API returned {len(data)} rows.")
        return pd.DataFrame(data)
    except Exception as e:
        print("NYC API query failed:", e)
        return pd.DataFrame()


def get_live_recent_311_examples(limit: int = 10) -> pd.DataFrame:
    """
    Pulls a small sample of recent 311 records.
    This is much faster than grouping or searching the full API.
    """
    params = {
        "$select": "created_date, complaint_type, descriptor, agency, borough, status, resolution_description",
        "$order": "created_date DESC",
        "$limit": limit
    }
    return nyc_api_get(params)


def search_live_311_examples(keyword: str, limit: int = 5) -> pd.DataFrame:
    """
    Safer keyword search:
    Pulls recent records first, then filters locally in Colab.
    """
    df = get_live_recent_311_examples(limit=200)

    if df.empty:
        return pd.DataFrame()

    keyword_upper = keyword.upper()

    for col in ["complaint_type", "descriptor", "resolution_description"]:
        if col not in df.columns:
            df[col] = ""

    mask = (
        df["complaint_type"].astype(str).str.upper().str.contains(keyword_upper, na=False) |
        df["descriptor"].astype(str).str.upper().str.contains(keyword_upper, na=False) |
        df["resolution_description"].astype(str).str.upper().str.contains(keyword_upper, na=False)
    )

    return df[mask].head(limit)


def get_live_agency_for_complaint(complaint_keyword: str, limit: int = 10) -> pd.DataFrame:
    """
    Looks for likely agencies from recent records only.
    If no result is found, the chatbot should fall back to the local agency map.
    """
    df = search_live_311_examples(complaint_keyword, limit=100)

    if df.empty or "agency" not in df.columns:
        return pd.DataFrame()

    agency_counts = (
        df.groupby(["complaint_type", "agency"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(limit)
    )

    return agency_counts


def get_live_top_complaint_types(limit: int = 10) -> pd.DataFrame:
    """
    Pulls recent records and calculates top complaint types locally.
    Avoids slow API-side grouping.
    """
    df = get_live_recent_311_examples(limit=500)

    if df.empty or "complaint_type" not in df.columns:
        return pd.DataFrame()

    top = (
        df["complaint_type"]
        .value_counts()
        .head(limit)
        .reset_index()
    )

    top.columns = ["complaint_type", "count"]
    return top

In [8]:
df_test = get_live_recent_311_examples(limit=5)
display(df_test)

Calling NYC API...
NYC API returned 5 rows.


,created_date,complaint_type,descriptor,agency,borough,status,resolution_description
0,2026-04-30T02:50:49.000,Street Condition,Pothole,DOT,QUEENS,Open,The Department of Transportation referred this...
1,2026-04-30T02:16:04.000,Street Condition,Pothole,DOT,QUEENS,Closed,The Department of Transportation determined th...
2,2026-04-30T02:06:22.000,Noise - Residential,Loud Television,NYPD,BRONX,In Progress,NaN
3,2026-04-30T02:06:21.000,Encampment,N/A,NYPD,BRONX,In Progress,NaN
4,2026-04-30T02:05:50.000,Animal-Abuse,Other (complaint details),NYPD,MANHATTAN,In Progress,NaN


## Retrieval helpers

These functions collect evidence from project outputs, sample data, and live NYC Open Data before sending it to the GenAI model.

In [9]:
# 7. Retrieve Context from Local Outputs and Live API

def table_preview(name: str, n: int = 5) -> str:
    if name not in tables or tables[name].empty:
        return ""
    return f"Table: {name}\n" + tables[name].head(n).to_markdown(index=False)

def detect_basic_topic(question: str) -> str:
    q = question.lower()
    if any(x in q for x in ["heat", "hot water", "landlord", "apartment"]):
        return "Heat"
    if any(x in q for x in ["noise", "loud", "music", "neighbor"]):
        return "Noise"
    if any(x in q for x in ["parking", "blocked driveway", "illegal parking"]):
        return "Parking"
    if any(x in q for x in ["trash", "garbage", "sanitation", "missed collection"]):
        return "Sanitation"
    if any(x in q for x in ["tree", "street light", "pothole", "sidewalk"]):
        return "Street"
    return ""

def retrieve_project_context(question: str) -> str:
    q = question.lower()
    parts = []

    if "top_findings_table.csv" in tables:
        parts.append(table_preview("top_findings_table.csv", 5))

    if any(x in q for x in ["top", "complaint", "volume", "common"]):
        for name in ["full_counts_complaint_type.csv", "top_problem_counts.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["borough", "brooklyn", "queens", "bronx", "manhattan", "staten"]):
        for name in ["full_counts_borough.csv", "borough_performance.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["agency", "department", "handled"]):
        for name in ["full_counts_agency.csv", "agency_performance.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["delay", "response", "long", "slow", "time"]):
        for name in ["sample_response_time_summary.csv", "sample_response_by_complaint_type.csv", "delay_rate_by_problem.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["model", "predict", "classification", "regression"]):
        for name in ["model_metrics.csv", "regression_model_metrics.csv", "classification_model_metrics.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))

    if any(x in q for x in ["repeat", "recurring"]):
        if "repeat_complaint_candidates.csv" in tables:
            parts.append(table_preview("repeat_complaint_candidates.csv", 10))

    topic = detect_basic_topic(question)
    if topic and not df_sample.empty:
        text_cols = [c for c in ["complaint_type", "descriptor", "agency", "borough"] if c in df_sample.columns]
        if text_cols:
            mask = pd.Series(False, index=df_sample.index)
            for c in text_cols:
                mask = mask | df_sample[c].astype(str).str.contains(topic, case=False, na=False)
            keep_cols = text_cols + [c for c in ["response_time_hours", "closed_same_day_flag"] if c in df_sample.columns]
            matches = df_sample.loc[mask, keep_cols].head(10)
            if not matches.empty:
                parts.append("Sample matching rows:\n" + matches.to_markdown(index=False))

    return "\n\n".join([p for p in parts if p])

def retrieve_live_context(question: str) -> str:
    q = question.lower()
    parts = []

    topic = detect_basic_topic(question)

    if any(x in q for x in ["top", "common", "highest volume", "most common"]):
        df = get_live_top_complaint_types(10)
        if not df.empty:
            parts.append("Live NYC Open Data top complaint types:\n" + df.to_markdown(index=False))

    if topic:
        examples = search_live_311_examples(topic, 5)
        if not examples.empty:
            parts.append(f"Recent NYC Open Data examples matching '{topic}':\n" + examples.to_markdown(index=False))

        agency = get_live_agency_for_complaint(topic, 10)
        if not agency.empty:
            parts.append(f"Common agencies for '{topic}' complaints:\n" + agency.to_markdown(index=False))

    return "\n\n".join(parts)


## Enhanced Call-Deflection Decision Layer

This section adds the refinement layer around the chatbot. It classifies the question, estimates confidence, routes emergencies or unclear cases, maps likely agencies, and creates structured metadata that can be saved with the demo transcript.


In [10]:
import re
from typing import Dict, List, Tuple

INTENT_RULES = {
    "noise_complaint": ["noise", "loud", "music", "party", "car alarm", "barking", "neighbor", "upstairs", "construction noise"],
    "heat_hot_water": ["heat", "hot water", "no heat", "radiator", "landlord", "apartment cold"],
    "housing_condition": ["mold", "leak", "ceiling", "stairs", "hallway", "building", "broken", "roach", "apartment"],
    "trash_sanitation": ["trash", "garbage", "recycling", "missed collection", "illegal dumping", "overflowing", "furniture", "sanitation"],
    "street_sidewalk": ["pothole", "sidewalk", "streetlight", "street light", "traffic signal", "traffic light", "street sign", "road"],
    "parking_vehicle": ["parking", "blocked driveway", "abandoned vehicle", "bus stop", "no plates", "vehicle", "car"],
    "rodents_pests": ["rat", "rats", "rodent", "pest", "roach", "mice", "mouse"],
    "parks_public_space": ["park", "playground", "tree", "branch", "dead tree", "graffiti", "bathroom"],
    "status_followup": ["status", "already filed", "complaint was closed", "closed", "reopen", "service request", "no action taken", "how long"],
    "analysis_question": ["most common", "highest", "leadership", "patterns", "borough", "agency", "delay", "longest", "focus", "analysis"]
}

EMERGENCY_TERMS = [
    "fire", "smoke", "gas smell", "smell gas", "carbon monoxide", "injured", "hurt", "medical emergency",
    "crime in progress", "breaking into", "assault", "shooting", "weapon", "immediate danger", "someone may be hurt"
]

DANGEROUS_311_TERMS = [
    "traffic light is out", "traffic signal is not working", "tree fell", "downed wire", "sinkhole", "dangerous", "hazard"
]

INTENT_TO_AGENCY = {
    "noise_complaint": "NYPD or DEP depending on the noise source",
    "heat_hot_water": "HPD",
    "housing_condition": "HPD",
    "trash_sanitation": "DSNY",
    "street_sidewalk": "DOT or DSNY depending on the issue",
    "parking_vehicle": "NYPD or DOT depending on the vehicle issue",
    "rodents_pests": "DOHMH",
    "parks_public_space": "NYC Parks",
    "status_followup": "NYC 311 / responsible agency listed on the service request",
    "analysis_question": "Analysis output, not a single agency"
}

INTENT_TO_ACTION = {
    "noise_complaint": "File a 311 noise complaint if the issue is ongoing and not an emergency.",
    "heat_hot_water": "File a 311 heat or hot water complaint and include building details.",
    "housing_condition": "File a 311 housing maintenance complaint with location and condition details.",
    "trash_sanitation": "File a 311 sanitation complaint and include the location and type of missed or improper collection.",
    "street_sidewalk": "File a 311 street or sidewalk condition complaint with the exact location.",
    "parking_vehicle": "File a 311 illegal parking or abandoned vehicle complaint if it is not an emergency.",
    "rodents_pests": "File a 311 rodent or pest complaint and include where the activity was seen.",
    "parks_public_space": "File a 311 parks or tree condition complaint with the park/location details.",
    "status_followup": "Use the service request number to check status through NYC 311 or the official 311 portal.",
    "analysis_question": "Use the project output tables to support leadership recommendations."
}


def classify_intent(question: str) -> Dict:
    q = question.lower().strip()

    emergency_hits = [term for term in EMERGENCY_TERMS if term in q]
    if emergency_hits:
        return {
            "intent": "emergency",
            "confidence": 1.0,
            "matched_terms": emergency_hits,
            "agency": "911",
            "recommended_action": "Call 911 immediately if there is danger, a crime in progress, fire, medical emergency, or possible injury.",
            "escalation": "911"
        }

    scores = []
    for intent, keywords in INTENT_RULES.items():
        hits = [kw for kw in keywords if kw in q]
        if hits:
            confidence = min(0.95, 0.45 + (0.15 * len(hits)))
            scores.append((intent, confidence, hits))

    if not scores:
        return {
            "intent": "unclear_general",
            "confidence": 0.30,
            "matched_terms": [],
            "agency": "NYC 311",
            "recommended_action": "Ask a clarifying question or contact 311 if the issue needs official city support.",
            "escalation": "311"
        }

    scores.sort(key=lambda x: x[1], reverse=True)
    intent, confidence, hits = scores[0]

    escalation = "none"
    if any(term in q for term in DANGEROUS_311_TERMS):
        escalation = "311_urgent_or_911_if_immediate_danger"
    elif confidence < 0.50:
        escalation = "311"

    return {
        "intent": intent,
        "confidence": round(confidence, 2),
        "matched_terms": hits,
        "agency": INTENT_TO_AGENCY.get(intent, "NYC 311"),
        "recommended_action": INTENT_TO_ACTION.get(intent, "Contact NYC 311 for guidance."),
        "escalation": escalation
    }


def get_common_issue_context(intent: str, top_n: int = 5) -> str:
    possible_tables = ["full_counts_complaint_type.csv", "top_problem_counts.csv", "sample_response_by_complaint_type.csv"]
    intent_terms = {
        "noise_complaint": ["noise"],
        "heat_hot_water": ["heat", "hot water"],
        "housing_condition": ["mold", "plumbing", "maintenance", "housing"],
        "trash_sanitation": ["trash", "garbage", "sanitation", "recycling", "dumping"],
        "street_sidewalk": ["pothole", "sidewalk", "street", "traffic", "signal"],
        "parking_vehicle": ["parking", "vehicle"],
        "rodents_pests": ["rodent", "rat", "pest"],
        "parks_public_space": ["park", "tree", "graffiti"],
    }.get(intent, [])

    if not intent_terms:
        return ""

    for table_name in possible_tables:
        if table_name in tables and not tables[table_name].empty:
            df = tables[table_name].copy()
            text_cols = [c for c in df.columns if any(x in c.lower() for x in ["complaint", "problem", "type", "descriptor"])]
            if not text_cols:
                continue
            mask = pd.Series(False, index=df.index)
            for c in text_cols:
                for term in intent_terms:
                    mask = mask | df[c].astype(str).str.contains(term, case=False, na=False)
            matches = df.loc[mask].head(top_n)
            if not matches.empty:
                return f"Common issue lookup from {table_name}:" + "\n" + matches.to_markdown(index=False)
    return ""


def format_decision_summary(meta: Dict) -> str:
    matched = ", ".join(meta.get("matched_terms", [])) if meta.get("matched_terms") else "none"
    return f"""
Structured decision metadata:
- Intent: {meta.get('intent')}
- Confidence: {meta.get('confidence')}
- Matched terms: {matched}
- Likely agency: {meta.get('agency')}
- Recommended action: {meta.get('recommended_action')}
- Escalation route: {meta.get('escalation')}
""".strip()

## GenAI response generator

This sends the user's question plus retrieved evidence to OpenAI and asks for a practical 311-style response.

In [11]:
# 8. Enhanced GenAI Response Layer

SYSTEM_INSTRUCTIONS = """
You are a NYC 311 GenAI call-deflection assistant prototype project.

Your role:
- Interpret the user's issue in plain English.
- Use the structured decision metadata as the primary routing signal.
- Suggest the likely 311 complaint category when evidence supports it.
- Mention the likely responsible agency when evidence supports it.
- Use historical/project evidence to explain expected patterns.
- Give practical next steps.
- Be clear that this prototype does not submit official 311 requests or check live request status.
- For emergencies, safety threats, crimes in progress, fire, gas smell, or medical issues, tell the user to call 911.

Required response format:
1. Likely issue
2. Recommended next step
3. Agency or service area
4. Notes from project or NYC 311 data, if available
5. Escalation guidance if needed

Rules:
- Do not invent exact response times unless provided in the evidence.
- If confidence is low or the issue is unclear, say the chatbot cannot determine the issue confidently and recommend contacting 311.
- If evidence is limited, say so.
- Keep responses concise but helpful.
- Use a calm, public-service tone.
"""

def genai_answer(question: str, use_live_api: bool = False, return_metadata: bool = False):
    decision_meta = classify_intent(question)

    if decision_meta["intent"] == "emergency":
        answer = """Likely issue: Emergency or possible immediate danger.

Recommended next step: Call 911 immediately. This chatbot prototype does not handle emergencies or submit official reports.

Agency or service area: 911 emergency services.

Escalation guidance: If anyone may be hurt, there is fire, a crime in progress, or a possible gas leak, use 911 rather than 311."""
        return (answer, decision_meta) if return_metadata else answer

    project_context = retrieve_project_context(question)
    common_issue_context = get_common_issue_context(decision_meta.get("intent", ""))
    live_context = retrieve_live_context(question) if use_live_api else ""

    evidence = f"""
{format_decision_summary(decision_meta)}

PROJECT OUTPUT CONTEXT:
{project_context if project_context else "No project context matched."}

COMMON ISSUE LOOKUP:
{common_issue_context if common_issue_context else "No common issue lookup matched."}

LIVE NYC OPEN DATA CONTEXT:
{live_context if live_context else "No live NYC Open Data context retrieved."}
"""

    user_prompt = f"""
User question:
{question}

Evidence:
{evidence}

Please answer as a NYC 311 GenAI call-deflection assistant prototype.
"""

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    answer = response.output_text
    return (answer, decision_meta) if return_metadata else answer


## Test the chatbot

Examples:

- "My apartment has no heat. What should I do?"
- "There is loud music next door. Is that a 311 issue?"
- "Who handles illegal parking?"
- "What are the most common 311 complaints?"
- "What should leadership focus on based on the analysis?"


In [12]:
# 9. Single Question Test

question = input("Ask the NYC 311 chatbot a question: ")
answer, meta = genai_answer(question, use_live_api=True, return_metadata=True)

print("\n--- Decision Metadata ---")
print(pd.DataFrame([meta]).to_markdown(index=False))

print("\n--- Chatbot Answer ---")
print(answer)

Ask the NYC 311 chatbot a question: "Who handles illegal parking?"
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.

--- Decision Metadata ---
| intent          |   confidence | matched_terms   | agency                                     | recommended_action                                                                   | escalation   |
|:----------------|-------------:|:----------------|:-------------------------------------------|:-------------------------------------------------------------------------------------|:-------------|
| parking_vehicle |          0.6 | ['parking']     | NYPD or DOT depending on the vehicle issue | File a 311 illegal parking or abandoned vehicle complaint if it is not an emergency. | none         |

--- Chatbot Answer ---
1. Likely issue: Illegal parking, such as parking violations including blocking hydrants, sidewalks, or posted parking sign violations.

2. Recommended next step: If the illegal parking is n

In [13]:
# 10. Multi-turn Chat Loop

while True:
    question = input("\nAsk a 311 question (or type quit): ")
    if question.lower().strip() in ["quit", "exit", "stop"]:
        print("Chat ended.")
        break

    try:
        answer, meta = genai_answer(question, use_live_api=True, return_metadata=True)
        print("\nDecision Metadata:")
        print(pd.DataFrame([meta]).to_markdown(index=False))
        print("\nAssistant:")
        print(answer)
    except Exception as e:
        print("Error:", e)
        print("If this is an API key or quota issue, check your OpenAI account/API key.")


Ask a 311 question (or type quit): My apartment has no heat. What should I do?
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.

Decision Metadata:
| intent         |   confidence | matched_terms       | agency   | recommended_action                                                   | escalation   |
|:---------------|-------------:|:--------------------|:---------|:---------------------------------------------------------------------|:-------------|
| heat_hot_water |         0.75 | ['heat', 'no heat'] | HPD      | File a 311 heat or hot water complaint and include building details. | none         |

Assistant:
1. Likely issue: No heat in your apartment, which is a heat or hot water complaint.
2. Recommended next step: File a 311 heat or hot water complaint with details about your building and apartment.
3. Agency or service area: NYC Department of Housing Preservation and Development (HPD).
4. Notes from project or NYC 311 data: Heat/hot wat

## Optional: Save a demo transcript

This saves sample chatbot interactions for your project materials.

In [14]:
# 11. Save Enhanced Demo Transcript and Evaluation Table

demo_questions = [
    "What can I do about loud music from my neighbor?",
    "There is construction noise late at night. Can I report that?",
    "A car alarm has been going off for hours. Is that a 311 issue?",
    "My upstairs neighbor is making noise every night. Who handles that?",
    "There is loud street noise from a party outside. What should I do?",
    "My apartment has no heat. What should I do?",
    "My landlord has not fixed hot water for three days.",
    "There is mold in my apartment. Is that a 311 complaint?",
    "My ceiling is leaking. Who do I contact?",
    "The building has broken stairs and no lights in the hallway.",
    "Trash was not picked up on my block. Can I report it?",
    "There is illegal dumping on the sidewalk.",
    "Someone left furniture outside for days.",
    "My recycling was missed. What should I do?",
    "There are overflowing garbage cans near my building.",
    "There is a pothole on my street.",
    "The sidewalk is cracked and dangerous.",
    "A streetlight is out.",
    "The traffic signal is not working.",
    "There is a damaged street sign.",
    "A car is blocking my driveway.",
    "There is an abandoned vehicle on my block.",
    "Someone parked in a bus stop.",
    "A vehicle has no plates and has been sitting for weeks.",
    "Can I report illegal parking through 311?",
    "There are rats in the alley.",
    "I saw rats near a restaurant.",
    "My building has a roach problem.",
    "There are rodents around trash outside.",
    "Who handles pest complaints?",
    "There is broken playground equipment.",
    "A park bathroom is dirty.",
    "There is graffiti in the park.",
    "A tree branch fell in the street.",
    "There is a dead tree near my building.",
    "I already filed a complaint. How do I check the status?",
    "My 311 complaint was closed but nothing was fixed.",
    "Why does my complaint say no action taken?",
    "Can I reopen a closed 311 request?",
    "How long does it take to resolve a pothole complaint?",
    "There is a fire in my building.",
    "Someone is breaking into a car.",
    "There is a gas smell in my apartment.",
    "A traffic light is out and cars are almost crashing.",
    "A tree fell on a car and someone may be hurt.",
    "My block is a mess. What should I report?",
    "Something smells bad outside.",
    "There is a problem with my building.",
    "I need help with a city issue.",
    "I do not know what category my complaint fits into.",
    "What are the most common 311 complaints?",
    "Which complaint types take the longest to resolve?",
    "Which borough has the most complaints?",
    "What patterns do you see in service request delays?",
    "What should leadership focus on based on the analysis?"
]

expected_intents = [
    "noise_complaint", "noise_complaint", "noise_complaint", "noise_complaint", "noise_complaint",
    "heat_hot_water", "heat_hot_water", "housing_condition", "housing_condition", "housing_condition",
    "trash_sanitation", "trash_sanitation", "trash_sanitation", "trash_sanitation", "trash_sanitation",
    "street_sidewalk", "street_sidewalk", "street_sidewalk", "street_sidewalk", "street_sidewalk",
    "parking_vehicle", "parking_vehicle", "parking_vehicle", "parking_vehicle", "parking_vehicle",
    "rodents_pests", "rodents_pests", "housing_condition", "rodents_pests", "rodents_pests",
    "parks_public_space", "parks_public_space", "parks_public_space", "parks_public_space", "parks_public_space",
    "status_followup", "status_followup", "status_followup", "status_followup", "status_followup",
    "emergency", "emergency", "emergency", "street_sidewalk", "emergency",
    "unclear_general", "unclear_general", "housing_condition", "unclear_general", "unclear_general",
    "analysis_question", "analysis_question", "analysis_question", "analysis_question", "analysis_question"
]

transcript = []

for q, expected_intent in zip(demo_questions, expected_intents):
    try:
        answer, meta = genai_answer(q, use_live_api=True, return_metadata=True)
        transcript.append({
            "question": q,
            "expected_intent": expected_intent,
            "predicted_intent": meta.get("intent"),
            "confidence": meta.get("confidence"),
            "agency": meta.get("agency"),
            "recommended_action": meta.get("recommended_action"),
            "escalation": meta.get("escalation"),
            "pass_fail": "PASS" if meta.get("intent") == expected_intent else "REVIEW",
            "answer": answer
        })
    except Exception as e:
        transcript.append({
            "question": q,
            "expected_intent": expected_intent,
            "predicted_intent": "ERROR",
            "confidence": None,
            "agency": None,
            "recommended_action": None,
            "escalation": None,
            "pass_fail": "ERROR",
            "answer": f"ERROR: {e}"
        })

transcript_df = pd.DataFrame(transcript)
transcript_df.to_csv("/content/nyc311_genai_chatbot_demo_transcript.csv", index=False)
transcript_df.to_csv("/content/nyc311_genai_chatbot_evaluation_table.csv", index=False)

display(transcript_df)
print("Saved: /content/nyc311_genai_chatbot_demo_transcript.csv")
print("Saved: /content/nyc311_genai_chatbot_evaluation_table.csv")


Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returned 200 rows.
Calling NYC API...
NYC API returne

,question,expected_intent,predicted_intent,confidence,agency,recommended_action,escalation,pass_fail,answer
0,What can I do about loud music from my neighbor?,noise_complaint,noise_complaint,0.90,NYPD or DEP depending on the noise source,File a 311 noise complaint if the issue is ong...,none,PASS,1. Likely issue: Loud music noise complaint fr...
1,There is construction noise late at night. Can...,noise_complaint,noise_complaint,0.75,NYPD or DEP depending on the noise source,File a 311 noise complaint if the issue is ong...,none,PASS,1. Likely issue: Construction noise occurring ...
2,A car alarm has been going off for hours. Is t...,noise_complaint,noise_complaint,0.60,NYPD or DEP depending on the noise source,File a 311 noise complaint if the issue is ong...,none,PASS,1. Likely issue: A vehicle noise complaint due...
3,My upstairs neighbor is making noise every nig...,noise_complaint,noise_complaint,0.90,NYPD or DEP depending on the noise source,File a 311 noise complaint if the issue is ong...,none,PASS,1. Likely issue: Noise complaint about a neigh...
4,There is loud street noise from a party outsid...,noise_complaint,noise_complaint,0.90,NYPD or DEP depending on the noise source,File a 311 noise complaint if the issue is ong...,none,PASS,1. Likely issue: Loud noise complaint due to a...
5,My apartment has no heat. What should I do?,heat_hot_water,heat_hot_water,0.75,HPD,File a 311 heat or hot water complaint and inc...,none,PASS,1. Likely issue: No heat in your apartment.\n2...
6,My landlord has not fixed hot water for three ...,heat_hot_water,heat_hot_water,0.75,HPD,File a 311 heat or hot water complaint and inc...,none,PASS,1. Likely issue: Lack of hot water in your apa...
7,There is mold in my apartment. Is that a 311 c...,housing_condition,housing_condition,0.75,HPD,File a 311 housing maintenance complaint with ...,none,PASS,1. Likely issue: Mold presence in an apartment...
8,My ceiling is leaking. Who do I contact?,housing_condition,housing_condition,0.75,HPD,File a 311 housing maintenance complaint with ...,none,PASS,1. Likely issue: Ceiling leak in a housing uni...
9,The building has broken stairs and no lights i...,housing_condition,housing_condition,0.95,HPD,File a 311 housing maintenance complaint with ...,none,PASS,1. Likely issue: Broken stairs and no lighting...


Saved: /content/nyc311_genai_chatbot_demo_transcript.csv
Saved: /content/nyc311_genai_chatbot_evaluation_table.csv
